<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit1/session-02-model-adapter/demo.ipynb)


# Session 2 — the live demo

**This is the notebook shown on screen, not the one you hand in.** Yours is
`notebook.ipynb` beside it, and it needs no key.

Everything here is the same course code you have. The difference is the lane:
some of these cells call a **real model**, which needs either Ollama running or
an API key in `.env`. If neither is set they still run — on `fake` — and the
notebook says what changes.

Run the cells in order. Each one is one idea.

In [1]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    LIVE = preflight(REPO_ROOT)  # the configured lane's client; FakeLLM whenever the lane is down

✅ Python 3.11 (need >= 3.11)
✅ kernel is the repo .venv
✅ corpus loads (6 documents)
✅ lane = ollama (llama3.2:1b at http://localhost:11434/v1)
ready. LIVE is the ollama lane.


## 1. The seam — one method, and nothing else is required

In [2]:
from bootcamp_agent.llm import FakeLLM

# An LLMClient is anything with complete(system, user) -> str. That is the
# entire contract: no base class, no registration, no vendor SDK.
fake = FakeLLM(responses={"agent": "An agent is a loop with a budget."})
print(fake.complete(system="You are concise.", user="What is an agent?"))

An agent is a loop with a budget.


## 2. Which lane am I on?

In [3]:
from bootcamp_agent.config import load_settings

settings = load_settings()
print(f"provider : {settings.provider}")
print(f"model    : {settings.model or '(the lane default)'}")
print(f"key set  : {settings.api_key is not None}")   # never the key itself

provider : ollama
model    : llama3.2:1b
key set  : False


## 3. The same question, through whatever lane is configured

Change one line in `.env` and run this again. Nothing else moves.

In [4]:
from bootcamp_agent.llm import get_client

client = get_client(settings)
print(client.complete(system="You are terse. One sentence.", user="What is an API?"))

An API is a set of defined rules that enables two parties, an **application** and an **interner**, to request for and send data between each other, usually over the internet.


## 4. Weak prompt vs project-aware prompt

The session's argument, in two calls. On `fake` both answers are identical —
that is FakeLLM being deterministic, not the prompts being equivalent.

In [5]:
WEAK = "add a search feature"
PROJECT_AWARE = (
    "read AGENTS.md, then propose a plan to add a tags filter to "
    "search_documents in src/bootcamp_agent/tools.py — plan only, no edits"
)

for label, prompt in (("WEAK", WEAK), ("PROJECT-AWARE", PROJECT_AWARE)):
    print(f"--- {label}")
    print(client.complete(system="You are a careful engineer.", user=prompt)[:600])
    print()

--- WEAK
I can add a basic search feature to our conversation. Here's an updated version of the conversation with a search bar at the top:

**Conversation**

* Previous message: You are a careful engineer.
* Search bar: Search for keywords (type here)

What would you like to talk about?

--- PROJECT-AWARE
Here's a proposed plan to add a tags filter to search_documents in `src/bootcamp_agent/tools.py`:

**Title:** Implementing a Tags Filter for Searching Documents

**Objective:** Add a tags filter to enable efficient searching of documents in `src/bootcamp_agent/tools.py` using PyShark.

**Approach:**

1. **Identify relevant modules and functions**: Review the existing code to identify modules and functions that use PyShark for streaming and processing data.
2. **Choose the correct PyShark filter**: Investigate relevant PyShark filters that might be suitable for this use case, such as `pyshark.f



## 5. Determinism is the property, not the answer

Ask twice. The fake agrees with itself every time; a real model usually does
not. That gap is why week 2 measures properties instead of words.

In [6]:
question = "Explain in one sentence what an agent is."
pair = [client.complete(system="You are sharp and concise.", user=question) for _ in range(2)]

print(f"lane {settings.provider}: same answer twice? {pair[0] == pair[1]}")
for reply in pair:
    print(f"   {reply[:110]}")

lane ollama: same answer twice? False
   An agent is a third-party entity that assumes control or responsibility for the management or direction of ano
   An agent is a person, business, or organization that assumes responsibility for the actions of a principal, co


## 6. Failing closed — a key that is missing

In [7]:
from bootcamp_agent.config import ConfigError, Settings

try:
    get_client(Settings(provider="anthropic", model=None, api_key=None, base_url=None))
except ConfigError as error:
    print(f"refused, as it should: {error}")
    # Note what is NOT in that message: the key. There is nothing to leak,
    # because nothing ever prints it.

refused, as it should: Provider 'anthropic' needs an API key in the environment; see .env.example. (The key itself is never printed.)


## 7. Failing closed — a provider nobody serves

In [8]:
try:
    load_settings(env={"BOOTCAMP_PROVIDER": "ollama"})
except ConfigError as error:
    print(f"refused at load time: {error}")

## 8. Failing closed — a deadline

The one the session builds to. A provider that never answers becomes a value
the caller can read, not an exception that escapes.

In [10]:
import time

from bootcamp_agent.agent import AgentResult, TraceEvent
from bootcamp_agent.ollama import OllamaError
from bootcamp_agent.schema import ResearchAnswer


class TimeoutLLM:
    """Any provider that runs past its deadline."""

    def complete(self, system: str, user: str) -> str:
        raise TimeoutError("no response within the deadline")


def answer_with_timeout(question: str) -> AgentResult:
    started = time.monotonic()
    try:
        text = TimeoutLLM().complete(system="You are concise.", user=question)
    except (TimeoutError, OllamaError) as error:
        waited = round((time.monotonic() - started) * 1000, 1)
        return AgentResult(
            answer=ResearchAnswer(
                answer="The model did not respond.",
                citations=(),
                confidence=0.0,
                needs_human_review=True,
            ),
            # The cause goes HERE, never in the answer: the adapter cannot tell
            # "too slow" from "no server", and a refusal that guesses is one
            # nobody can debug.
            trace=(TraceEvent(kind="llm_call", detail=f"provider=timeoutllm waited={waited}ms: {error}"),),
        )
    return AgentResult(answer=ResearchAnswer(answer=text, citations=(), confidence=0.5,
                                             needs_human_review=False), trace=())


result = answer_with_timeout("How does chunking work in RAG?")
print(result.answer.answer)
print(f"review needed: {result.answer.needs_human_review}   citations: {list(result.answer.citations)}")
for event in result.trace:
    print(f"  trace[{event.kind}] {event.detail}")

The model did not respond.
review needed: True   citations: []
  trace[llm_call] provider=timeoutllm waited=0.0ms: no response within the deadline


## 9. The bar it has to clear

Both checks, on the function above: the graded one, and the bonus.

In [11]:
from bootcamp_agent.bonus import bonus
from bootcamp_agent.checks import check

check("ch02-e4", answer_with_timeout)
bonus("ch02", answer_with_timeout)

✅ ch02-e4 passed
✅ bonus ch02 passed — above the floor.


True

## 10. And the course can answer questions about itself

No key, no network, no assistant — the pages, retrieved in this kernel.

In [12]:
from bootcamp_agent.coach import coach

coach("which sessions need a coding assistant?")

--- Give the course to your assistant  [unit0/ask-your-assistant]
- It is already there — `from bootcamp_agent.coach import coach`
- Ask — `coach("what is AGENTS.md for?")`
- It prints the passages **and the page id each came from**. A page id is something you can open; a paraphrase is not
- It only knows the weeks in **your** clone. Ask about an unpublished week and it says so instead of inventing one
- No key, no network, no model. It quotes pages, it does not write prose

## In a terminal

The coach is its own small program, and `uvx` runs it straight from GitHub —
nothing to install, nothing to clone, no PyPI.

```bash
uvx --from "git+https://github.com/Gecko-Academy/gecko-ai-coach" \
  ai-coach ask "which sessions need a coding assistant" --pages ./units/en
```

--- Instructions are code  [unit1/session-01-assistant-configuration/concepts-2]
Read this repository's `AGENTS.md` against that table. Every row is present.
The "Do not" rows sit under **Coding rules** and **Safety**: no 

## What to take away

- An `LLMClient` is one method. Everything else is a detail of one lane.
- A lane change moves **wording, latency and cost** — never permissions,
  grading or redaction.
- Every failure leaves as a **value the caller can read**, and the cause lives
  in the trace, not in the answer.

Your own notebook is `notebook.ipynb`. Three of its four exercises already run;
the fourth is section 7, and it is the one you write.